In [1]:
import pandas as pd
import numpy as np
import torch
# import matplotlib.pyplot as plt

In [ ]:
## We get an embedding vector of d = 2**n
## We got exactly n grades levels each with dimension \binom{n}{b}
## Previous approaches (See DeCal and Keci) only use grade-0 and grade-1, which is a small fraction of the total dimension.
## We will use all grades, and we will show that this is a better approach.



########## first step: Identify all the grades and their dimensions ###########

# First get n = log2(d)

#---- Grade 0 ----
# Cl^{0}(V,q) = {v_0}, dim(Cl^{0}(V,q)) = 1

#---- Grade 1 ----
# Cl^{1}(V,q) = <e_1, e_2, ..., e_n>, dim(Cl^{1}(V,q)) = n

#---- Grade 2 ----
# Cl^{2}(V,q) = <e_1e_2, e_1e_3, ..., e_{n-1}e_n>, dim(Cl^{2}(V,q)) = \binom{n}{2}

#---- Grade b ----
# Cl^{b}(V,q) = <e_{i_1}e_{i_2}...e_{i_b} | 1 <= i_1 < i_2 < ... < i_b <= n>, dim(Cl^{b}(V,q)) = \binom{n}{b}

#---- Grade n ----
# Cl^{n}(V,q) = <e_1e_2...e_n>, dim(Cl^{n}(V,q)) = 1

# Cl(V,q) = \bigoplus_{b=0}^{n} Cl^{b}(V,q), dim(Cl(V,q)) = 2^n
# v \in Cl(V,q) can be expressed as v = \sum_{b=0}^{n} v_b, where v_b \in Cl^{b}(V,q)
# v_b represents the grade-b component of v.

In [37]:
s = 0
l = []
n = 10
for p in range(n+1):
    for q in range(n+1):
        for r in range(n+1):
            if p+q+r==n:
                s+=1
                l.append((p,q,r))

print(s)
l

66


[(0, 0, 10),
 (0, 1, 9),
 (0, 2, 8),
 (0, 3, 7),
 (0, 4, 6),
 (0, 5, 5),
 (0, 6, 4),
 (0, 7, 3),
 (0, 8, 2),
 (0, 9, 1),
 (0, 10, 0),
 (1, 0, 9),
 (1, 1, 8),
 (1, 2, 7),
 (1, 3, 6),
 (1, 4, 5),
 (1, 5, 4),
 (1, 6, 3),
 (1, 7, 2),
 (1, 8, 1),
 (1, 9, 0),
 (2, 0, 8),
 (2, 1, 7),
 (2, 2, 6),
 (2, 3, 5),
 (2, 4, 4),
 (2, 5, 3),
 (2, 6, 2),
 (2, 7, 1),
 (2, 8, 0),
 (3, 0, 7),
 (3, 1, 6),
 (3, 2, 5),
 (3, 3, 4),
 (3, 4, 3),
 (3, 5, 2),
 (3, 6, 1),
 (3, 7, 0),
 (4, 0, 6),
 (4, 1, 5),
 (4, 2, 4),
 (4, 3, 3),
 (4, 4, 2),
 (4, 5, 1),
 (4, 6, 0),
 (5, 0, 5),
 (5, 1, 4),
 (5, 2, 3),
 (5, 3, 2),
 (5, 4, 1),
 (5, 5, 0),
 (6, 0, 4),
 (6, 1, 3),
 (6, 2, 2),
 (6, 3, 1),
 (6, 4, 0),
 (7, 0, 3),
 (7, 1, 2),
 (7, 2, 1),
 (7, 3, 0),
 (8, 0, 2),
 (8, 1, 1),
 (8, 2, 0),
 (9, 0, 1),
 (9, 1, 0),
 (10, 0, 0)]

In [25]:
p,q,r = 1,1,0

def eta(i,p,q,r):
    if 1<=i<=p:
        return 1
    elif p+1<=i<=p+q:
        return -1
    elif p+q+1<=i<=p+q+r:
        return 0

# print(eta(3))

def gamma(K):
    k = len(K)
    k_2 = k*(k+1)//2
    return (-1)**(k_2)

print(gamma({1,2}))

def delta(I,J):
    return I.union(J) - I.intersection(J)

print(delta({1,2,3},{2,3,4}))

-1
{1, 4}


In [4]:
# dim(B) = 2^{p+q+r} = 2^{0+1+1} = 4, hence the basis of Cl(V,q) is given by
# B = {1, e_1, e_2, e_1e_2 }, 

n = p + q + r
print(n)
I, J, K =  range(1,n+1), range(1,n+1), range(1,n+1) #Indexes for h,r and t

# Clifford multiplication of two vectors in Cl(V,q)
#  \mathbf{h}\mathbf{r} =
# \sum_{I,J}
# h_Ir_J
# \sigma(I,J)
# \left(
#     \prod_{k\in I\cap J}\eta_k
# \right)
# \mathbf{e}_{I\triangle J}.


def clifford_mult(h, r):
    n = len(h)
    result = np.zeros(n)
    I = set()
    for i in range(n):
        J = set()
        for j in range(n):
            I.add(i+1)
            J.add(j+1)
            K = delta(I,J)
            sigma = gamma(K)
            eta_prod = np.prod([eta(k) for k in I.intersection(J)])
            result[list(K)[0]-1] += h[i] * r[j] * sigma * eta_prod
    return result


2


In [39]:
I = set()
for i in range(n):
    J = set()
    for j in range(n):
        I.add(i+1)
        J.add(j+1)

In [41]:

def clifford_mult(h, r):
    """
    Geometric product of two full multivectors in Cl_{p,q,r}.

    Parameters
    ----------
    h : np.ndarray
        Coefficients of the first multivector.
        Length must be 2**n.

    r : np.ndarray
        Coefficients of the second multivector.
        Length must be 2**n.

    eta : np.ndarray
        Signature coefficients [e_1^2, ..., e_n^2],
        where each entry is +1, -1, or 0.

    Returns
    -------
    result : np.ndarray
        Coefficients of h * r.
    """

    d = len(h)

    assert len(r) == d
    assert d > 0 and (d & (d - 1)) == 0

    n = int(np.log2(d))

    eta = [eta(i) for i in range(1, n + 1)]

    assert len(eta) == n

    result = np.zeros(d)

    for I in range(d):
        for J in range(d):

            # Symmetric difference
            K = I ^ J

            # Sign from reordering basis vectors
            sigma = blade_sign(I, J, n)

            # Product of eta_k for repeated basis vectors
            common = I & J

            eta_prod = 1.0

            for k in range(n):
                if common & (1 << k):
                    eta_prod *= eta[k]

            result[K] += h[I] * r[J] * sigma * eta_prod

    return result

{1, 2}

In [6]:
def blade_sign(I, J, n):
    """
    Computes the sign sigma(I,J) induced by
    reordering e_I e_J into canonical order.
    """

    swaps = 0

    for i in range(n):

        # If e_i occurs in the left blade
        if I & (1 << i):

            # Count basis vectors in J with smaller index
            lower_bits = J & ((1 << i) - 1)

            swaps += lower_bits.bit_count()

    return -1 if swaps % 2 else 1

In [29]:

# ─────────────────────────────────────────────────────────────────────────────
# Full DeCaL scoring function
#
#   f(h, r, t) = sum_{I,J}  h_I r_J  sigma(I,J)
#                            [prod_{k in I∩J} eta_k]
#                            t_{I△J}
#
# Embeddings are indexed by bitmasks over {0, …, n-1}:
#   bit i set  ↔  basis vector e_{i+1} is present.
#
# Bitmask ordering for n=2  (Cl_{1,1,0}):
#   0 → scalar (∅),  1 → e1,  2 → e2,  3 → e12
#
# Bitmask ordering for n=3  (Cl_{1,1,1}):
#   0 → scalar,  1 → e1,  2 → e2,  3 → e12,
#   4 → e3,      5 → e13, 6 → e23, 7 → e123
# ─────────────────────────────────────────────────────────────────────────────

def full_decal_score(h, r, t, eta_arr):
    """
    Full DeCaL triple score.

    f(h, r, t) = sum_{I,J}  h_I r_J  blade_sign(I,J,n)
                             [prod_{k in I∩J} eta_k]
                             t[I^J]

    Parameters
    ----------
    h, r, t  : np.ndarray of length 2^n   (bitmask-ordered coefficients)
    eta_arr  : np.ndarray of length n      (eta_arr[i] = eta_{i+1})

    Returns
    -------
    float : the triple score
    """
    d = len(h)
    n = int(np.log2(d))
    score = 0.0

    for I in range(d):
        for J in range(d):
            K      = I ^ J                 # symmetric difference  I△J
            sig    = blade_sign(I, J, n)   # reordering sign  sigma(I,J)
            common = I & J                 # intersection  I∩J

            # prod_{k in I∩J} eta_k
            eta_int = 1.0
            for k in range(n):
                if common & (1 << k):
                    eta_int *= eta_arr[k]

            score += h[I] * r[J] * sig * eta_int * t[K]

    return score


In [32]:

# ─────────────────────────────────────────────────────────────────────────────
# Test A: Cl_{1,1,0}  (n=2, eta=[+1,-1])
#
# Bitmask: 0=scalar, 1=e1, 2=e2, 3=e12
#
# Explicit score (eq. cl110-coefficient-score):
#   f = (h0 r0 + h1 r1 − h2 r2 + h12 r12)   t0
#     + (h0 r1 + h1 r0 + h2 r12 − h12 r2)   t1
#     + (h0 r2 + h2 r0 + h1 r12 − h12 r1)   t2
#     + (h0 r12 + h12 r0 + h1 r2 − h2 r1)   t12
# ─────────────────────────────────────────────────────────────────────────────

def explicit_score_cl110(h, r, t):
    h0, h1, h2, h12 = h[:4]
    r0, r1, r2, r12 = r[:4]
    t0, t1, t2, t12 = t[:4]
    return (
          (h0*r0  + h1*r1   - h2*r2   + h12*r12) * t0
        + (h0*r1  + h1*r0   + h2*r12  - h12*r2)  * t1
        + (h0*r2  + h2*r0   + h1*r12  - h12*r1)  * t2
        + (h0*r12 + h12*r0  + h1*r2   - h2*r1)   * t12
    )


# ─────────────────────────────────────────────────────────────────────────────
# Test B: Cl_{1,1,1}  (n=3, eta=[+1,-1,0])
#
# Bitmask: 0=scalar, 1=e1, 2=e2, 3=e12, 4=e3, 5=e13, 6=e23, 7=e123
#
# The t0..t12 coefficients are identical to Cl_{1,1,0} because eta_3=0
# kills all I∩J terms involving e3.
# The remaining coefficients are computed from blade_sign + eta_int:
#
#   t3:   h0 r3 + h3 r0 + h1 r13 - h13 r1 - h2 r23 + h23 r2 + h12 r123 + h123 r12
#   t13:  h0 r13 + h13 r0 + h1 r3 - h3 r1 + h2 r123 + h123 r2 - h12 r23 + h23 r12
#   t23:  h0 r23 + h23 r0 + h2 r3 - h3 r2 + h1 r123 + h123 r1 - h12 r13 + h13 r12
#   t123: h0 r123 + h123 r0 + h1 r23 + h23 r1 - h2 r13 - h13 r2 + h12 r3 + h3 r12
#
# Key: pairs like (h12,r123) and (h123,r12) contributing to t3 both get sign +1
# because blade_sign × eta_int = (−1) × (−1) = +1 for those pairs.
# ─────────────────────────────────────────────────────────────────────────────

def explicit_score_cl111(h, r, t):
    h0, h1, h2, h12, h3, h13, h23, h123 = h
    r0, r1, r2, r12, r3, r13, r23, r123 = r
    t0, t1, t2, t12, t3, t13, t23, t123 = t

    # t0 … t12: identical to Cl_{1,1,0} (eta_3=0 kills extra cross terms)
    f_base = explicit_score_cl110(h, r, t)

    f_e3 = (
        h0*r3    + h3*r0
      + h1*r13   - h13*r1
      - h2*r23   + h23*r2
      + h12*r123 + h123*r12    # both +: blade_sign×eta_int = (−1)×(−1) = +1
    ) * t3

    f_e13 = (
        h0*r13   + h13*r0
      + h1*r3    - h3*r1
      + h2*r123  + h123*r2     # both +: same reason
      - h12*r23  + h23*r12
    ) * t13

    f_e23 = (
        h0*r23   + h23*r0
      + h2*r3    - h3*r2
      + h1*r123  + h123*r1     # both +
      - h12*r13  + h13*r12
    ) * t23

    f_e123 = (
        h0*r123  + h123*r0
      + h1*r23   + h23*r1      # both +
      - h2*r13   - h13*r2      # both −
      + h12*r3   + h3*r12
    ) * t123

    return f_base + f_e3 + f_e13 + f_e23 + f_e123


# ─────────────────────────────────────────────────────────────────────────────
# Run tests
# ─────────────────────────────────────────────────────────────────────────────

eta_cl110 = np.array([1.0, -1.0])
eta_cl111 = np.array([1.0, -1.0, 0.0])

np.random.seed(42)
N = 1_000

for label, d, eta_arr, explicit_fn in [
    ("Cl_{1,1,0}", 4, eta_cl110, explicit_score_cl110),
    ("Cl_{1,1,1}", 8, eta_cl111, explicit_score_cl111),
]:
    passed = True
    for trial in range(N):
        h_vec = np.random.randn(d)
        r_vec = np.random.randn(d)
        t_vec = np.random.randn(d)

        f_gen = full_decal_score(h_vec, r_vec, t_vec, eta_arr)
        f_exp = explicit_fn(h_vec, r_vec, t_vec)

        if not np.isclose(f_gen, f_exp, atol=1e-10):
            print(f"FAIL [{label}] trial {trial}: general={f_gen:.8f}  explicit={f_exp:.8f}")
            passed = False
            break

    if passed:
        print(f"✓  All {N} tests PASSED for {label}")
        print(f"   Last example:  general f = {f_gen:.10f}   |diff| = {abs(f_gen - f_exp):.2e}")
        print()


✓  All 1000 tests PASSED for Cl_{1,1,0}
   Last example:  general f = 0.7442149975   |diff| = 4.44e-16

✓  All 1000 tests PASSED for Cl_{1,1,1}
   Last example:  general f = 2.7909267974   |diff| = 4.44e-16



In [34]:

def make_eta(p, q, r):
    """
    Build the eta array for Cl_{p,q,r}.
    eta_arr[i] = eta_{i+1} = square of the (i+1)-th generator:
      +1 for the first p  generators  (indices 1..p)
      -1 for the next  q  generators  (indices p+1..p+q)
       0 for the last  r  generators  (indices p+q+1..p+q+r)
    """
    return np.array(
        [1.0] * p + [-1.0] * q + [0.0] * r
    )

# Examples
for (pp, qq, rr) in [(1, 1, 0), (1, 1, 1), (0, 2, 0), (2, 0, 0), (3, 2, 1)]:
    arr = make_eta(pp, qq, rr)
    print(f"Cl_{{{pp},{qq},{rr}}}  →  eta_arr = {arr}")


Cl_{1,1,0}  →  eta_arr = [ 1. -1.]
Cl_{1,1,1}  →  eta_arr = [ 1. -1.  0.]
Cl_{0,2,0}  →  eta_arr = [-1. -1.]
Cl_{2,0,0}  →  eta_arr = [1. 1.]
Cl_{3,2,1}  →  eta_arr = [ 1.  1.  1. -1. -1.  0.]


In [ ]:

import torch
import torch.nn as nn
import math


# ═════════════════════════════════════════════════════════════════════════════
#  Precomputed geometric-product table
# ═════════════════════════════════════════════════════════════════════════════

def _blade_sign_torch(I: int, J: int, n: int) -> int:
    """Sign sigma(I,J) from bubble-sort reordering (pure Python, called once at init)."""
    swaps = 0
    for i in range(n):
        if I & (1 << i):
            lower = J & ((1 << i) - 1)
            swaps += bin(lower).count('1')
    return -1 if swaps % 2 else 1


def build_product_table(p: int, q: int, r: int) -> tuple[torch.LongTensor, torch.FloatTensor]:
    """
    Precompute the geometric product table for Cl_{p,q,r}.

    Returns
    -------
    K_table   : LongTensor  (d, d)   – bitmask of the output blade  I△J
    coeff_table: FloatTensor (d, d)  – scalar coefficient  sigma(I,J) * prod_{k in I∩J} eta_k
    """
    n = p + q + r
    d = 1 << n                         # 2^n

    # eta array: +1 for first p bits, -1 for next q, 0 for last r
    eta = [1.0] * p + [-1.0] * q + [0.0] * r

    K_table    = torch.zeros(d, d, dtype=torch.long)
    coeff_table = torch.zeros(d, d, dtype=torch.float32)

    for I in range(d):
        for J in range(d):
            K = I ^ J
            sig = _blade_sign_torch(I, J, n)

            # prod_{k in I∩J} eta_k
            common = I & J
            eta_int = 1.0
            for k in range(n):
                if common & (1 << k):
                    eta_int *= eta[k]

            K_table[I, J]    = K
            coeff_table[I, J] = sig * eta_int

    return K_table, coeff_table


# ═════════════════════════════════════════════════════════════════════════════
#  Full-DeCaL model
# ═════════════════════════════════════════════════════════════════════════════

class FullDeCaL(nn.Module):
    """
    Full Clifford Knowledge Graph Embedding model.

    Scoring function (no conjugation factor):

        f(h, r, t) = sum_{I,J}  h_I r_J  sigma(I,J) [prod_{k in I∩J} eta_k]  t_{I△J}

    Embedding layout
    ----------------
    Each entity / relation embedding vector has dimension  d = 2^n * re
    where  n = p + q + r  and  re = embedding_dim // 2^n.
    The embedding is split into  2^n  blocks of size  re, one per blade.

    Parameters
    ----------
    num_entities   : int
    num_relations  : int
    embedding_dim  : int   – must satisfy  embedding_dim % 2^n == 0
    p, q, r        : int   – Clifford signature  (p + q + r == n)
    """

    def __init__(
        self,
        num_entities:  int,
        num_relations: int,
        embedding_dim: int,
        p: int,
        q: int,
        r: int,
    ):
        super().__init__()
        n  = p + q + r
        d  = 1 << n          # number of blades  2^n
        assert embedding_dim % d == 0, (
            f"embedding_dim ({embedding_dim}) must be divisible by 2^n={d}  (n=p+q+r={n})"
        )
        self.n  = n
        self.d  = d                          # number of blades
        self.re = embedding_dim // d         # per-blade dimension
        self.p, self.q, self.r = p, q, r

        self.entity_embeddings   = nn.Embedding(num_entities,  embedding_dim)
        self.relation_embeddings = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.entity_embeddings.weight)
        nn.init.xavier_uniform_(self.relation_embeddings.weight)

        # Precompute product table (registered as buffers → moves with .to(device))
        K_table, coeff_table = build_product_table(p, q, r)
        self.register_buffer('K_table',    K_table)    # (d, d)  long
        self.register_buffer('coeff_table', coeff_table)  # (d, d)  float

    # ─────────────────────────────────────────────────────────────────────────
    #  Internal helpers
    # ─────────────────────────────────────────────────────────────────────────

    def _split(self, emb: torch.Tensor) -> torch.Tensor:
        """
        (B, d*re)  →  (B, d, re)   split embedding into blade components.
        """
        B = emb.size(0)
        return emb.view(B, self.d, self.re)

    def _hr_product(
        self,
        h: torch.Tensor,   # (B, d, re)
        r: torch.Tensor,   # (B, d, re)
    ) -> torch.Tensor:
        """
        Compute the geometric product coefficients  z_K = sum_{I,J: I△J=K}  c_IJ h_I r_J
        for every blade K, batched over (B, re).

        Returns z : (B, d, re)
        """
        # w[b, I, J, e] = coeff[I,J] * h[b,I,e] * r[b,J,e]
        # Then accumulate into z[b, K, e]  where K = K_table[I,J]

        # Step 1: outer product over blades, weighted by coeff_table
        # h: (B, d, re), r: (B, d, re)
        # hr[b, I, J, e] = h[b,I,e] * r[b,J,e]  →  (B, d, d, re)
        hr = torch.einsum('bie,bje->bije', h, r)          # (B, d, d, re)

        # Step 2: multiply by coefficient table  (d, d) → broadcast over B, re
        hr = hr * self.coeff_table.unsqueeze(0).unsqueeze(-1)  # (B, d, d, re)

        # Step 3: scatter-add into z[b, K, e]
        # K_table: (d, d)   – output blade index
        d, re = self.d, self.re
        B = h.size(0)
        z = torch.zeros(B, d, re, device=h.device, dtype=h.dtype)
        K_flat = self.K_table.view(-1)                    # (d*d,)
        hr_flat = hr.view(B, d * d, re)                   # (B, d*d, re)
        z.scatter_add_(1, K_flat.unsqueeze(0).unsqueeze(-1).expand(B, -1, re), hr_flat)
        # z: (B, d, re)
        return z

    # ─────────────────────────────────────────────────────────────────────────
    #  KvsAll  –  returns (B, num_entities)
    # ─────────────────────────────────────────────────────────────────────────

    def forward_k_vs_all(self, x: torch.LongTensor) -> torch.FloatTensor:
        """
        x : (B, 2)  – columns are [head_idx, relation_idx]
        Returns scores of shape (B, num_entities).

        f(h, r, t) = sum_K  z_K  dot  t_K       (inner product over re dim, sum over blades)
        where z_K = sum_{I△J=K} c_IJ h_I r_J
        """
        head_idx, rel_idx = x[:, 0], x[:, 1]

        h = self._split(self.entity_embeddings(head_idx))    # (B, d, re)
        r = self._split(self.relation_embeddings(rel_idx))   # (B, d, re)

        z = self._hr_product(h, r)                           # (B, d, re)

        # All entity embeddings  (E, d, re)
        T = self._split(self.entity_embeddings.weight)       # (E, d, re)

        # score[b, e] = sum_{k, dim}  z[b,k,dim] * T[e,k,dim]
        scores = torch.einsum('bkr,ekr->be', z, T)           # (B, E)
        return scores

    # ─────────────────────────────────────────────────────────────────────────
    #  NegSample  –  returns (B,)
    # ─────────────────────────────────────────────────────────────────────────

    def forward_triples(self, x: torch.LongTensor) -> torch.FloatTensor:
        """
        x : (B, 3)  – columns are [head_idx, relation_idx, tail_idx]
        Returns scores of shape (B,).
        """
        head_idx, rel_idx, tail_idx = x[:, 0], x[:, 1], x[:, 2]

        h = self._split(self.entity_embeddings(head_idx))    # (B, d, re)
        r = self._split(self.relation_embeddings(rel_idx))   # (B, d, re)
        t = self._split(self.entity_embeddings(tail_idx))    # (B, d, re)

        z = self._hr_product(h, r)                           # (B, d, re)

        # score[b] = sum_{k, dim}  z[b,k,dim] * t[b,k,dim]
        scores = torch.einsum('bkr,bkr->b', z, t)            # (B,)
        return scores

    def forward(self, x: torch.LongTensor) -> torch.FloatTensor:
        if x.shape[1] == 2:
            return self.forward_k_vs_all(x)
        return self.forward_triples(x)


In [40]:
2 << 3

16

In [ ]:

import time

# ─────────────────────────────────────────────────────────────────────────────
# Correctness test: forward_triples must agree with the reference numpy scorer
# for Cl_{1,1,0} and Cl_{1,1,1}
# ─────────────────────────────────────────────────────────────────────────────

def check_correctness(p, q, r, num_entities=32, num_relations=8, batch=16, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n  = p + q + r
    d  = 1 << n
    re = 4                  # per-blade dimension
    emb_dim = d * re

    eta_arr = np.array([1.0]*p + [-1.0]*q + [0.0]*r)

    model = FullDeCaL(num_entities, num_relations, emb_dim, p, q, r)
    model.eval()

    # random batch of triples
    heads = torch.randint(0, num_entities,  (batch,))
    rels  = torch.randint(0, num_relations, (batch,))
    tails = torch.randint(0, num_entities,  (batch,))

    with torch.no_grad():
        scores_torch = model.forward_triples(
            torch.stack([heads, rels, tails], dim=1)
        ).numpy()

    # Reference: numpy full_decal_score on the same weight tensors
    E = model.entity_embeddings.weight.detach().numpy()   # (num_entities, d*re)
    R = model.relation_embeddings.weight.detach().numpy()

    scores_np = np.array([
        full_decal_score(E[heads[i]], R[rels[i]], E[tails[i]], eta_arr)
        for i in range(batch)
    ])

    max_err = np.max(np.abs(scores_torch - scores_np))
    ok = max_err < 1e-4
    print(f"Cl_{{{p},{q},{r}}}  emb_dim={emb_dim}  max|err|={max_err:.2e}  {'✓ PASS' if ok else '✗ FAIL'}")
    return ok

all_ok = True
for (p, q, r) in [(1, 1, 0), (1, 1, 1), (2, 1, 0), (0, 3, 0)]:
    all_ok &= check_correctness(p, q, r)

print()
if all_ok:
    print("All correctness tests PASSED.")


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Speed benchmark: KvsAll throughput (triples/sec)
# ─────────────────────────────────────────────────────────────────────────────

def benchmark_kvsall(p, q, r, num_entities=40_000, num_relations=500,
                     emb_dim=None, batch=256, repeats=5):
    n = p + q + r
    d = 1 << n
    if emb_dim is None:
        emb_dim = d * 16   # 16-dim per blade

    model = FullDeCaL(num_entities, num_relations, emb_dim, p, q, r)
    model.eval()

    x = torch.stack([
        torch.randint(0, num_entities,  (batch,)),
        torch.randint(0, num_relations, (batch,)),
    ], dim=1)

    # warm-up
    with torch.no_grad():
        for _ in range(3):
            _ = model.forward_k_vs_all(x)

    times = []
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.perf_counter()
            _ = model.forward_k_vs_all(x)
            times.append(time.perf_counter() - t0)

    ms = min(times) * 1e3
    throughput = batch / min(times)
    print(f"Cl_{{{p},{q},{r}}}  emb_dim={emb_dim:5d}  |E|={num_entities}  "
          f"batch={batch}  →  {ms:6.1f} ms/batch  ({throughput:,.0f} triples/s)")

print("KvsAll throughput benchmark")
print("─" * 72)
benchmark_kvsall(1, 1, 0)   # n=2, d=4,  emb_dim=64
benchmark_kvsall(1, 1, 1)   # n=3, d=8,  emb_dim=128
benchmark_kvsall(2, 1, 0)   # n=3, d=8,  emb_dim=128
benchmark_kvsall(1, 2, 1)   # n=4, d=16, emb_dim=256
